In [1]:
import pandas as pd

games = pd.read_csv("https://github.com/nflverse/nfldata/raw/master/data/games.csv")
played = games[games["result"].notna()].copy()
played.shape

(7443, 46)

In [ ]:
# Avewrage combined points per season
played.groupby("season")["total"].mean().round(1)

season
1999    41.7
2000    41.1
2001    40.5
2002    43.7
2003    41.8
2004    43.1
2005    41.1
2006    41.5
2007    43.4
2008    44.0
2009    43.2
2010    44.3
2011    44.5
2012    45.8
2013    46.9
2014    45.3
2015    45.5
2016    45.7
2017    43.6
2018    46.6
2019    45.7
2020    49.5
2021    46.1
2022    44.0
2023    43.8
2024    46.0
2025    46.0
2026    46.4
Name: total, dtype: float64

In [3]:
# Domes vs. Outdoors
played.groupby("roof")["total"].mean().round(1)

roof
closed      47.1
dome        46.9
open        46.7
outdoors    43.3
Name: total, dtype: float64

In [4]:
# Several stats at once with agg
played.groupby("roof").agg(
    n_games=("game_id", "count"),
    avg_total=("total", "mean"),
    avg_home_margin=("result", "mean"),
).round(1)

,n_games,avg_total,avg_home_margin
roof,,,
closed,640,47.1,1.1
dome,1227,46.9,2.5
open,136,46.7,1.6
outdoors,5440,43.3,2.4


In [5]:
# Make a new column then group by it
played["home_win"] = played["result"] > 0

home_record = (
    played.groupby("home_team")["home_win"]
    .mean()
    .sort_values(ascending=False)
    .round(3)
)
home_record.head(10)

home_team
NE     0.722
GB     0.699
BAL    0.675
PIT    0.674
SEA    0.658
MIN    0.651
KC     0.648
IND    0.646
DEN    0.633
PHI    0.607
Name: home_win, dtype: float64

In [6]:
# Merge, joining a second table
teams = pd.read_csv("https://github.com/nflverse/nflverse-data/releases/download/teams/teams_colors_logos.csv")
teams[["team_abbr", "team_name", "team_conf", "team_division"]].head()

,team_abbr,team_name,team_conf,team_division
0,ARI,Arizona Cardinals,NFC,NFC West
1,ATL,Atlanta Falcons,NFC,NFC South
2,BAL,Baltimore Ravens,AFC,AFC North
3,BUF,Buffalo Bills,AFC,AFC East
4,CAR,Carolina Panthers,NFC,NFC South


In [7]:
home_df = home_record.reset_index()

home_named = home_df.merge(
    teams[["team_abbr", "team_name", "team_division"]],
    left_on="home_team",
    right_on="team_abbr",
    how="left",
)
home_named.head(10)

,home_team,home_win,team_abbr,team_name,team_division
0,NE,0.722,NE,New England Patriots,AFC East
1,GB,0.699,GB,Green Bay Packers,NFC North
2,BAL,0.675,BAL,Baltimore Ravens,AFC North
3,PIT,0.674,PIT,Pittsburgh Steelers,AFC North
4,SEA,0.658,SEA,Seattle Seahawks,NFC West
5,MIN,0.651,MIN,Minnesota Vikings,NFC North
6,KC,0.648,KC,Kansas City Chiefs,AFC West
7,IND,0.646,IND,Indianapolis Colts,AFC South
8,DEN,0.633,DEN,Denver Broncos,AFC West
9,PHI,0.607,PHI,Philadelphia Eagles,NFC East


In [10]:
# Home-field advantage by season: find the home win rate for each season.
# Which recent season is the lowest, and can you guess why? (Think about what was missing from stadiums that year.)

played["home_win"] = played["result"] > 0

home_record = (
    played.groupby("season")["home_win"]
    .mean()
    .sort_values()
    .round(3)
)
home_record.head(10)

season
2020    0.498
2026    0.509
2021    0.516
2019    0.521
2025    0.533
2006    0.539
2015    0.543
2024    0.547
2010    0.554
2001    0.556
Name: home_win, dtype: float64

In [22]:
# Best road offenses: which 5 teams have averaged the most points per game as the away team since 1999?
# (Hint: group by away_team, average away_score.)

avg_away_points = (
    played.groupby("away_team")["away_score"]
    .mean()
    .sort_values(ascending=False)
    .round(1)
)
avg_away_points.head(10)

# Show the number of games played along with the average using agg
played.groupby("away_team").agg(
    games=("away_score", "count"),
    avg_pts=("away_score", "mean"),
).sort_values("avg_pts", ascending=False).round(1).head(10)

,games,avg_pts
away_team,,
LA,98,24.9
NE,238,24.8
GB,242,24.3
NO,231,23.6
IND,239,23.4
PHI,237,23.4
KC,233,23.2
SD,149,22.7
LAC,84,22.3


In [21]:
# Merge, then group (the challenge): merge teams onto played to add each home team's conference (team_conf),
# then find the home win rate for the AFC vs. NFC.

played["home_win"] = played["result"] > 0
home_teams = played.merge(
    teams[["team_abbr", "team_name", "team_conf"]],
    left_on = "home_team",
    right_on = "team_abbr",
    how = "left"
)

conf_home_win_rate = (
    home_teams.groupby("team_conf")["home_win"]
    .mean()
    .sort_values(ascending=False)
    .round(3)
)
conf_home_win_rate

team_conf
AFC    0.570
NFC    0.554
Name: home_win, dtype: float64